# 02 · From HF to `relax.frontend.nn`

TVM does not trace the PyTorch model. The model is **written again** with `tvm.relax.frontend.nn`
(`fast_dllm_relax/model/cuda.py`), and each decoding mode is exported as its own Relax function. This
notebook shows:
- what the rewrite changes;
- what the export spec asks for (shapes, dtypes, symbolic dims);
- the first Relax IR, before any compiler pass;
- how the HF weights map onto the Relax parameters.

**Needs:** TVM only. The last section reads the cached checkpoint's safetensors header (shapes only,
no weights loaded). Runs in seconds.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
from transformers import AutoConfig
from fast_dllm_relax.common import DEFAULT_MODEL_NAME, DEFAULT_MODEL_REVISION
from fast_dllm_relax.export_utils import make_relax_config
from fast_dllm_relax.model import cuda as M

hf_cfg = AutoConfig.from_pretrained(DEFAULT_MODEL_NAME, revision=DEFAULT_MODEL_REVISION, trust_remote_code=True)
cfg = make_relax_config(hf_cfg)
print_table([{"FastDLLMRelaxConfig": k, "real": v, "tiny (IR notebooks)": getattr(tiny_config(True), k)}
             for k, v in vars(cfg).items()])

FastDLLMRelaxConfig  real    tiny (IR notebooks)
-------------------  ------  -------------------
vocab_size           151936  64                 
hidden_size          1536    256                
intermediate_size    8960    512                
num_hidden_layers    28      2                  
num_attention_heads  12      4                  
num_key_value_heads  2       2                  
head_dim             128     128                
rms_norm_eps         1e-06   1e-06              
rope_theta           1e+06   1e+06              
context_window_size  32768   256                
tie_word_embeddings  True    True               
bd_size              32      32                 
pad_token_id         151645  0                  


## What the rewrite changes

The module tree matches HF's (`model.layers[i].self_attn...`), so parameter names line up. The
differences:

| Change | Why |
|---|---|
| `q/k/v_proj` → one `qkv_proj`; `gate/up_proj` → one `gate_up_proj` | One GEMM instead of three or two. The weights are concatenated at export (`remap_params`). |
| RMSNorm normalizes in fp32, casts, *then* multiplies by the weight | Copies HF's precision order; TVM's built-in RMSNorm multiplies in fp32. |
| SiLU computed in fp32 | Copies PyTorch's CUDA kernel; TVM's `silu` in bf16 drifts. |
| Attention without cache: explicit fp32 `QKᵀ` → +bias → softmax → fp32 `PV` | Copies SDPA's fp32 accumulation; the mask becomes an additive −1e9 bias input. |
| Attention against a cache: call out to `fast_dllm.attn_inplace` | A VM → Python callback into PyTorch SDPA/flash-attn that also writes the block's K/V into the preallocated cache in place (notebook 06). |
| Tied `get_logits` multiplies by `embed_tokens.weight` | No second copy of the vocab × hidden matrix. |
| `get_logits` returns fp32 | Softmax/argmax then run on fp32 logits (HF: bf16). |

In [3]:
show_source(M.FastDLLMRMSNorm.forward)
show_source(M._silu_fp32)
show_source(M.FastDLLMRelaxForCausalLM.get_logits)

# 3rdparty/tvm/python/tvm/relax/frontend/nn/subroutine.py:283
    def forward(self, x: Tensor) -> Tensor:
        input_dtype = x.dtype
        xf = op.astype(x, "float32")
        variance = _mean(op.square(xf), axis=-1, keepdims=True)
        xf = op.multiply(xf, _rsqrt(op.add(variance, Tensor.from_scalar(self.epsilon, "float32"))))
        x_cast = op.astype(xf, input_dtype)
        return op.multiply(self.weight, x_cast)
# fast_dllm_relax/model/cuda.py:358
def _silu_fp32(x: Tensor, name: str = "silu_fp32") -> Tensor:
    """SiLU computed via fp32 intermediate precision, matching PyTorch's CUDA SiLU
    kernel (opmath convention for reduced-precision dtypes). TVM's op.silu computes
    sigmoid/exp directly in bf16 via the CUDA `hexp` intrinsic — an empirically
    measured divergence of MAE ~0.0021/call from HF.
    """
    input_dtype = x.dtype
    xf = op.astype(x, "float32")
    out = op.multiply(xf, op.sigmoid(xf))
    return op.astype(out, input_dtype)
# fast_dllm_relax/model/c

## The export spec: what TVM asks for

TVM compiles *functions*, each with a fixed signature. `get_default_spec()` declares ten:
- one per runtime mode;
- `replace_position` (0/8/16/24) baked into four separate partial functions;
- symbolic dims (`seq_len`, `prefix_len`, `block_len`) wherever a length varies at runtime.

`"$": {"param_mode": "packed"}` means every function takes all the weights as one trailing tuple.

In [4]:
model = M.FastDLLMRelaxForCausalLM(cfg, dtype="bfloat16", fused_qkv=True, fused_gate_up=True)
rows = []
for fn, args in model._default_spec_raw().items():
    for arg, spec in args.items():
        if arg != "$":
            rows.append({"function": fn, "arg": arg, "shape": spec.shape, "dtype": spec.dtype})
print_table(rows)

function                    arg             shape                                dtype   
--------------------------  --------------  -----------------------------------  --------
embed                       input_ids       ['seq_len']                          int32   
forward_hidden              input_embed     [1, 'seq_len', 1536]                 bfloat16
forward_hidden              attention_bias  [1, 12, 'seq_len', 'seq_len']        float32 
get_logits                  hidden_states   [1, 'seq_len', 1536]                 bfloat16
build_prefix_cache          prefix_embed    [1, 'prefix_len', 1536]              bfloat16
build_prefix_cache          prefix_bias     [1, 12, 'prefix_len', 'prefix_len']  float32 
forward_cached              active_embed    [1, 'block_len', 1536]               bfloat16
forward_cached              cached_k        [28, 1, 2, 'prefix_len', 128]        bfloat16
forward_cached              cached_v        [28, 1, 2, 'prefix_len', 128]        bfloat16
forward_ca

How these map to HF's `forward(...)` calls:

| Relax function | HF call it replaces |
|---|---|
| `embed` | `embed_tokens(ids)` |
| `forward_hidden` | `forward()` with no cache and a block-causal mask (the stateless first block) |
| `build_prefix_cache` | prefill `forward(use_cache=True, update_past_key_values=True)`: writes rows `[0, P)` of the cache |
| `forward_cached` | block-append and refinement `forward(past_key_values, update_past_key_values=…)`: writes the block's K/V at rows `[P, P+32)`; the runtime commits them (append) or not (refinement) |
| `forward_block_partial_rp{0,8,16,24}` | `forward(use_block_cache=True, replace_position=rp)` on 8 tokens: overwrites rows `[P+rp, P+rp+8)` |
| `get_logits` | `lm_head(hidden)`, as fp32 |

## Export: the first Relax IR

The rest of the notebook uses the tiny config (2 layers, hidden 256) so the IR stays readable. The
structure is identical to the real model's; only the constants differ.

In [5]:
tiny, mod, named = export_model(tiny_config(tied=True), dtype="float16")
list_functions(mod)

function                    kind           lines
--------------------------  -------------  -----
build_prefix_cache          Relax (entry)  217  
embed                       Relax (entry)  28   
forward_block_partial_rp0   Relax (entry)  238  
forward_block_partial_rp16  Relax (entry)  242  
forward_block_partial_rp24  Relax (entry)  238  
forward_block_partial_rp8   Relax (entry)  242  
forward_cached              Relax (entry)  220  
forward_cached_block        Relax (entry)  220  
forward_hidden              Relax (entry)  215  
get_logits                  Relax (entry)  30   


In [6]:
show_ir(mod, "get_logits")

# from tvm.script import tirx as T
# from tvm.script import relax as R

@R.function
def get_logits(hidden_states: R.Tensor((1, "seq_len", 256), dtype="float16"), packed_params: R.Tuple(R.Tensor((64, 256), dtype="float16"), R.Tensor((1024, 256), dtype="float16"), R.Tensor((1024,), dtype="float16"), R.Tensor((256, 512), dtype="float16"), R.Tensor((1024, 256), dtype="float16"), R.Tensor((256, 512), dtype="float16"), R.Tensor((256,), dtype="float16"), R.Tensor((256,), dtype="float16"), R.Tensor((1024, 256), dtype="float16"), R.Tensor((1024,), dtype="float16"), R.Tensor((256, 512), dtype="float16"), R.Tensor((1024, 256), dtype="float16"), R.Tensor((256, 512), dtype="float16"), R.Tensor((256,), dtype="float16"), R.Tensor((256,), dtype="float16"), R.Tensor((256,), dtype="float16"))) -> R.Tensor((1, "seq_len", 64), dtype="float32"):
    seq_len = T.int64()
    R.func_attr({"num_input": 1})
    with R.dataflow():
        model_embed_tokens_weight2: R.Tensor((64, 256), dtype="float16") = packed_

`get_logits` takes the hidden states plus `packed_params`, a tuple of every weight in export order.
The dataflow block holds high-level Relax ops (`R.permute_dims`, `R.matmul`, `R.astype`). Nothing is
a kernel yet; notebook 04 follows how they become kernels.

In [7]:
for name in ["embed", "forward_hidden", "build_prefix_cache", "forward_cached", "forward_block_partial_rp8"]:
    f = mod[name]
    print(name)
    for p in f.params[:-1]:
        print(f"    {p.name_hint:<14} {p.struct_info}")
    print(f"    packed_params  tuple of {len(f.params[-1].struct_info.fields)} tensors")
    print(f"    -> {f.ret_struct_info}")

embed
    input_ids      R.Tensor((seq_len,), dtype="int32")
    packed_params  tuple of 16 tensors
    -> R.Tensor((seq_len, 256), dtype="float16")
forward_hidden
    input_embed    R.Tensor((1, seq_len, 256), dtype="float16")
    attention_bias R.Tensor((1, 4, seq_len, seq_len), dtype="float32")
    packed_params  tuple of 16 tensors
    -> R.Tensor((1, seq_len, 256), dtype="float16")
forward_cached
    active_embed   R.Tensor((1, block_len, 256), dtype="float16")
    cached_k       R.Tensor((2, 1, 2, prefix_len, 128), dtype="float16")
    cached_v       R.Tensor((2, 1, 2, prefix_len, 128), dtype="float16")
    cache_offset   R.Tensor((), dtype="int32")
    packed_params  tuple of 16 tensors
    -> R.Tuple(R.Tensor((1, block_len, 256), dtype="float16"), R.Tensor((2, 1, 2, prefix_len + block_len, 128), dtype="float16"), R.Tensor((2, 1, 2, prefix_len + block_len, 128), dtype="float16"))
forward_cached_block
    active_embed   R.Tensor((1, block_len, 256), dtype="float16")
    cached_k 

The KV cache is one preallocated buffer, `[layers, 1, cap, kv_heads, head_dim]` with symbolic
capacity `cap`, passed in and written **in place**; `prefix_len` is a host int (`R.Shape([prefix_len])`).
`build_prefix_cache` returns the written buffers; the cached forwards return only hidden states.

Inside `forward_cached`, each layer hands its query, the block's K/V and the whole cache to the
Python callback through `call_inplace_packed`. The cache outputs alias the inputs and are threaded to
the next layer, which orders the writes. There is no slice, concat or transpose of the cache:

In [8]:
show_ir(mod, "forward_cached", grep=r"strided_slice|concat|attn_inplace|permute_dims", max_lines=30)

        lv102: R.Tensor((1, 1, 2, prefix_len, 128), dtype="float16") = R.strided_slice(cached_k, (R.prim_value(0),), (R.prim_value(0),), (R.prim_value(1),), (R.prim_value(1),), assume_inbound=False)
    ...
        lv103: R.Tensor((1, 1, 2, prefix_len, 128), dtype="float16") = R.strided_slice(cached_v, (R.prim_value(0),), (R.prim_value(0),), (R.prim_value(1),), (R.prim_value(1),), assume_inbound=False)
    ...
        permute_dims53: R.Tensor((256, 1024), dtype="float16") = R.permute_dims(model_layers_0_self_attn_qkv_proj_weight5, axes=None)
        matmul33: R.Tensor((1, block_len, 1024), dtype="float16") = R.matmul(mul81, permute_dims53, out_dtype="void")
    ...
        lv114: R.Tensor((block_len, 128), dtype="float32") = R.concat((lv113, lv113), axis=-1)
    ...
        concat20: R.Tensor((1, block_len, 4, 128), dtype="float16") = R.concat((mul83, split_025), axis=-1)
        mul84: R.Tensor((1, block_len, 4, 128), dtype="float16") = R.multiply(concat20, sin6)
    ...
        conca

## From HF weights to Relax parameters

The checkpoint's safetensors header gives every HF tensor's name and shape without loading any weights.
`remap_params` (`fast_dllm_relax/export_utils.py`) applies three rules:
- concatenate q/k/v and gate/up along the output dim;
- copy everything else by name;
- skip `lm_head`, which the tied model no longer has.

In [9]:
from huggingface_hub import try_to_load_from_cache
from safetensors import safe_open

st = try_to_load_from_cache(DEFAULT_MODEL_NAME, "model.safetensors", revision=DEFAULT_MODEL_REVISION)
with safe_open(st, "pt") as f:
    hf_shapes = {k: tuple(f.get_slice(k).get_shape()) for k in f.keys()}
relax_shapes = {k: tuple(int(d) for d in p.shape) for k, p in model.named_parameters()}

def sources(name):
    if "qkv_proj" in name:
        return [name.replace("qkv_proj", p) for p in ("q_proj", "k_proj", "v_proj")]
    if "gate_up_proj" in name:
        return [name.replace("gate_up_proj", p) for p in ("gate_proj", "up_proj")]
    return [name]

rows = []
for name, shape in relax_shapes.items():
    if ".layers." in name and ".layers.0." not in name:
        continue
    src = sources(name)
    how = src[0] if len(src) == 1 else "concat(" + ", ".join(s.split(".")[-2] for s in src) + ")"
    rows.append({"relax param": name, "shape": shape, "from HF": how,
                 "HF shapes": " + ".join(str(hf_shapes.get(s)) for s in src)})
print_table(rows)
print(f"\n(layers 1..27 repeat layer 0)  HF tensors in file: {len(hf_shapes)}   Relax params: {len(relax_shapes)}")
print("lm_head.weight in checkpoint file:", "lm_head.weight" in hf_shapes)

relax param                                     shape           from HF                                         HF shapes                               
----------------------------------------------  --------------  ----------------------------------------------  ----------------------------------------
model.embed_tokens.weight                       (151936, 1536)  model.embed_tokens.weight                       (151936, 1536)                          
model.layers.0.self_attn.qkv_proj.weight        (2048, 1536)    concat(q_proj, k_proj, v_proj)                  (1536, 1536) + (256, 1536) + (256, 1536)
model.layers.0.self_attn.qkv_proj.bias          (2048,)         concat(q_proj, k_proj, v_proj)                  (1536,) + (256,) + (256,)               
model.layers.0.self_attn.o_proj.weight          (1536, 1536)    model.layers.0.self_attn.o_proj.weight          (1536, 1536)                            
model.layers.0.mlp.gate_up_proj.weight          (17920, 1536)   concat(gate_proj, 

In [10]:
from pathlib import Path
artifact_names = Path("artifacts/fast_dllm_relax_bf16/fast_dllm_relax_nn_param_names.txt").read_text().splitlines()
print("export order matches the bf16 artifact's param_names.txt:", artifact_names == list(relax_shapes))
missing = [n for n in relax_shapes for s in sources(n) if s not in hf_shapes]
print("Relax params with no HF source:", missing or "none")

export order matches the bf16 artifact's param_names.txt: True
Relax params with no HF source: none


## Takeaways: what TVM asks of the model
- **Pure functions with declared shapes.** Every runtime mode is a separate function. Anything that
  varies at runtime is a symbolic dim; anything that changes control flow (`replace_position`) is a
  compile-time constant, hence the four `rp*` variants.
- **Functional caches.** Caches go in as inputs and come out as new tensors; nothing is updated in place.
- **Weights as inputs.** `packed_params` means one compiled artifact works with any weights of the
  same shapes, and the weights live in a separate `.npz`.
- **Precision is explicit.** Every cast in the IR is one written in `model/cuda.py`. Matching HF is a
  matter of writing them in HF's order.
- Next: **03 · Numerical parity**.